<div class="alert alert-block alert-info">
<b>Migrated</b> from v0.9.0-beta to the market-generic / generic-energy-model AeroMAPS
(see <code>markets_equilibriums.yaml</code>, <code>energy_carriers_equilibriums.yaml</code>,
<code>config_equilibriums.yaml</code> next to this notebook). Model (a) -- historical
price/volume calibration -- is now AeroMAPS's shipped default behaviour
(`PassengerAircraftMarginalCost` + `RPKElasticity`), so it no longer needs a custom
class; models (b)/(c)/(d) are ported to the new market-generic cost I/O.
</div>

## Cost-demand feedback: data exploitation and models testing

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import seaborn as sns
import gemseo as gm
import pandas as pd
import numpy as np
from typing import Tuple

%matplotlib widget

from aeromaps import create_process
from aeromaps.models.base import AeroMAPSModel
from aeromaps.models.air_transport.air_traffic.rpk_market import RPKElasticity
from aeromaps.utils.functions import custom_logger_config

custom_logger_config(gm.configure_logger())

# load traffic data
df = pd.read_excel(
    "./../../../../resources/cost_data/IATA_cost_data.xlsx", sheet_name="Exploration", index_col=0
)

df = df.head(22)
df_t = df.T
df_t

In [ ]:
# inflation rates https://data.worldbank.org/indicator/FP.CPI.TOTL.ZG?end=2024&locations=1W&start=1981&view=chart

# Inflation rates for 1981-2024
inflation_rates = [
    12.44243689,
    10.22172714,
    8.669271598,
    8.080320173,
    6.807566558,
    5.822666962,
    5.710119385,
    7.113406529,
    6.923905037,
    8.063460909,
    8.996938735,
    7.636108523,
    7.144587069,
    10.24793556,
    9.077380952,
    6.526095694,
    5.554129889,
    5.097291493,
    3.041946671,
    3.433515634,
    3.836572621,
    2.90799857,
    3.025045263,
    3.517999031,
    4.107250707,
    4.267174634,
    4.810237043,
    8.949953354,
    2.860448559,
    3.326344634,
    4.82239636,
    3.725326661,
    2.651673429,
    2.354490528,
    1.443857193,
    1.605539174,
    2.254276519,
    2.442583297,
    2.206073058,
    1.905663587,
    3.475403203,
    7.922048831,
    5.870102688,
    3.01447577,
    3,
]


years = list(range(1980, 2026))

cpi_values = [100]

# Build CPI series
for rate in inflation_rates:
    prev_cpi = cpi_values[-1]
    new_cpi = prev_cpi * (1 + rate / 100)
    cpi_values.append(new_cpi)

# Align CPI series with years
cpi_series = pd.Series(cpi_values, index=years)

# Reference CPI for 2020
deflator_2020 = cpi_series.loc[2020] / cpi_series

df_t["deflator_2020"] = deflator_2020.loc[df_t.index]

df_t["Real avg price (incl. anx), $/RPK"] = (
    df_t["Avg price (incl. anx), $/RPK"] * df_t["deflator_2020"]
)
df_t["Real avg cost, $/RPK"] = df_t["Avg cost, $/RPK"] * df_t["deflator_2020"]

In [ ]:
fig = plt.figure(figsize=(10, 6))

plt.plot(df_t.index, df_t["Avg price (incl. anx), $/RPK"], label="Price (Nominal)", color="#cb3629")
plt.plot(df_t.index, df_t["Avg cost, $/RPK"], label="Cost (Nominal)", color="#092054")

plt.plot(
    df_t.index,
    df_t["Real avg price (incl. anx), $/RPK"],
    label="Price (Real)",
    color="#cb3629",
    ls="--",
)
plt.plot(df_t.index, df_t["Real avg cost, $/RPK"], label="Cost (Real)", color="#092054", ls="--")

plt.xlabel("")
plt.ylim(
    0,
)
plt.legend()

In [ ]:
df_t.drop([2020, 2022, 2021], inplace=True)

Compute a corrected price with actual IATA markup but AeroMAPS 2019 total cost

In [ ]:
# Everything below needs AeroMAPS's own 2019 (last-historical-year) total cost per
# RPK as an anchor. Rather than hardcode the value from the old model version, fetch
# it fresh from the current one: create the process this notebook will reuse for
# model (a), run it once at the yaml's fallback `initial_airfare_per_rpk`, and read
# back `total_cost_per_rpk_without_extra_tax` for the last historical year. Only the
# elasticity calibration below (`initial_airfare_per_rpk`) depends on that fallback
# value, so this first pass is otherwise representative of the final run.
process_a = create_process(configuration_file="./config_equilibriums.yaml")

# BAU overrides: frozen technology from 2025 (near-zero efficiency & operational
# gains), a common world-average 2019 load factor, and the paper's carbon-tax /
# kerosene-price trajectory (the kerosene ramp itself lives in
# energy_carriers_equilibriums.yaml).
process_a.parameters.short_range_energy_per_ask_dropin_fuel_gain_reference_years = []
process_a.parameters.short_range_energy_per_ask_dropin_fuel_gain_reference_years_values = [
    0.0000000001
]
process_a.parameters.medium_range_energy_per_ask_dropin_fuel_gain_reference_years = []
process_a.parameters.medium_range_energy_per_ask_dropin_fuel_gain_reference_years_values = [
    0.0000000001
]
process_a.parameters.long_range_energy_per_ask_dropin_fuel_gain_reference_years = []
process_a.parameters.long_range_energy_per_ask_dropin_fuel_gain_reference_years_values = [
    0.0000000001
]
process_a.parameters.operations_final_gain = 0.0000000001  # [%]
process_a.parameters.operations_start_year = 2025
process_a.parameters.operations_duration = 25.0
process_a.parameters.short_range_load_factor_end_year = 82.39931200000001
process_a.parameters.medium_range_load_factor_end_year = 82.39931200000001
process_a.parameters.long_range_load_factor_end_year = 82.39931200000001
process_a.parameters.carbon_tax_reference_years = [2020, 2034, 2035, 2050]
process_a.parameters.carbon_tax_reference_years_values = [0, 0, 300, 300]

process_a.compute()
intial_total_cost_per_rpk_without_extra_tax = process_a.data["vector_outputs"][
    "total_cost_per_rpk_without_extra_tax"
][process_a.parameters.prospection_start_year - 1]
intial_total_cost_per_rpk_without_extra_tax

In [ ]:
df_t["normalised_price"] = df_t["Real avg price (incl. anx), $/RPK"] / 1.14 - (
    df_t["Real avg cost, $/RPK"] / 1.14 - intial_total_cost_per_rpk_without_extra_tax
)
euro_dollar = 1.14

In [ ]:
df_t["normalised_price"]

## Models comparison

Two options are explored as potential supply models
 - a) Consider the initial corrected price as a baseline, use baseline growth rpk to correct as to have user's default growth when 2019 conditions remain trhough the scenario
 - b) Consider a fixed supply function determined on average past conditions, demand adjusted to match with equilibrium price as to have user's default growth when 2019 conditions remain trhough the scenario
 - c) Fixed (average) markup
 - d) Fixed (average) margin

In both a) and b) cases the idea is to have a linear inverse suppy function f(Q), which can be entirely determined base on initial (or historical) cost and price values

## a) Historical price/volume callibration
The idea is to keep the historical price to callibrate the supply function, but to adjust the function to match the desired RPK.
Technically, it means that teh whole industry scales up without affecting the cost structure to match an increasing exogenous demand.

This calibration is now AeroMAPS's **built-in default**: `PassengerAircraftMarginalCost`
implements exactly this inverse supply function, anchored on
`global.elasticity.initial_airfare_per_rpk` in `markets_equilibriums.yaml`. No custom
class is needed for option (a) any more -- we only need to feed it the calibration
derived below.

In [ ]:
average_normalised_price = df_t["normalised_price"].mean()

In [ ]:
average_normalised_price

In [ ]:
# Second (and final) pass: feed the freshly-derived calibration into the stock
# model and recompute. process_a already carries every other override from above.
process_a.parameters.initial_airfare_per_rpk = average_normalised_price
process_a.compute()

## b) Historical function + future volume callibration
The idea is to keep the historical supply function, before tuning the isoelastic demand to match that point (it's rthe only degree of freedom we have as supply is fixed).
Technically, it means that the industry increase its marginal cost (BEFORE CONSIDERING EFFICIENCY) to match an increasing demand. As initial demand stays the same with an increasing "initial" price, it becomes less price sensitive.

In [ ]:
a = (
    2
    * (df_t["normalised_price"] - intial_total_cost_per_rpk_without_extra_tax)
    / df_t["RPKs, billion"]
)

In [ ]:
b = 2 * intial_total_cost_per_rpk_without_extra_tax - df_t["normalised_price"]

In [ ]:
x = np.linspace(3000, 12000, 500)
Y = np.array([slope * x + intercept for slope, intercept in zip(a, b)])

# Mean and standard deviation across functions
y_mean = Y.mean(axis=0)
y_std = Y.std(axis=0)

mean_slope = np.mean(a)
mean_intercept = np.mean(b)

sns.set(style="whitegrid")
line_color = sns.color_palette("deep")[0]
fill_color = sns.color_palette("deep")[0]
individual_line_color = sns.color_palette("gray")[2]

plt.figure(figsize=(10, 6))

for y in Y:
    plt.plot(x, y, color=individual_line_color, alpha=0.2, linewidth=1)

# Plot mean line
plt.plot(x, y_mean, color=line_color, linewidth=2, label="Mean Function")

# Fill +/-1 std deviation
plt.fill_between(
    x, y_mean - y_std, y_mean + y_std, color=fill_color, alpha=0.2, label="+/-1 std deviation"
)

equation_text = rf"$y = {mean_slope:.2e} \cdot x + {mean_intercept:.2e}$"
plt.text(
    0.98,
    0.95,
    equation_text,
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment="top",
    horizontalalignment="right",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="lightgrey", linewidth=1.2),
)

x_marker = df_t["RPKs, billion"]
y_marker = df_t["normalised_price"]
palette = sns.color_palette("viridis", len(a))
for i, (year, valx, valy) in enumerate(zip(df_t.index, x_marker, y_marker)):
    plt.scatter(valx, valy, color=palette[i], edgecolor="black", zorder=5)
    plt.text(valx + 151, valy - 0.0001, str(year), fontsize=10, va="center")

plt.xlabel("x", fontsize=12)
plt.ylabel("y", fontsize=12)
plt.title("Inverse offer function callibration, option b", fontsize=14)
plt.ylim(0.08, 0.11)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

The function seems to match well historical data on which it is callibrated.

In [ ]:
class PassengerAircraftMarginalCostB(AeroMAPSModel):
    """Fixed historical inverse supply function (option b): the linear
    cost=f(rpk) calibrated above (`mean_slope`, `mean_intercept`) is held fixed;
    only the demand side reacts through `RPKElasticityB`."""

    def __init__(self, name="passenger_aircraft_marginal_cost", fleet_model=None, *args, **kwargs):
        super().__init__(name=name, *args, **kwargs)

    def compute(
        self,
        rpk: pd.Series,
        rpk_no_elasticity: pd.Series,
        total_cost_per_rpk_without_extra_tax: pd.Series,
        total_extra_tax_per_rpk: pd.Series,
        total_subsidy_per_rpk: pd.Series,
    ) -> Tuple[pd.Series, pd.Series, pd.Series, pd.Series]:
        """
        Returns
        -------
        marginal_cost_per_rpk
        airfare_per_rpk_true
        airfare_per_rpk
        airfare_per_rpk_base
            Counterfactual airfare at `rpk_no_elasticity` (no elasticity, no policy
            change) -- the moving reference price `RPKElasticityB` divides by.
        """
        intial_total_cost_per_rpk_without_extra_tax = total_cost_per_rpk_without_extra_tax[
            self.prospection_start_year - 1
        ]

        b = mean_intercept
        a = mean_slope
        proj = slice(self.prospection_start_year, self.end_year)

        # For latter update replace total cost by the step component of the marginal cost.
        marginal_cost_per_rpk = (
            a * rpk.loc[proj] / 1e9
            + b
            + total_cost_per_rpk_without_extra_tax.loc[proj]
            - intial_total_cost_per_rpk_without_extra_tax
        )
        marginal_cost_per_rpk_base = a * rpk_no_elasticity.loc[proj] / 1e9 + b

        airfare_per_rpk_true = (
            marginal_cost_per_rpk + total_extra_tax_per_rpk - total_subsidy_per_rpk
        )
        airfare_per_rpk = airfare_per_rpk_true
        airfare_per_rpk_base = (
            marginal_cost_per_rpk_base
            + total_extra_tax_per_rpk[self.prospection_start_year - 1]
            - total_subsidy_per_rpk[self.prospection_start_year - 1]
        )

        self.df.loc[:, "marginal_cost_per_rpk"] = marginal_cost_per_rpk
        self.df.loc[:, "airfare_per_rpk"] = airfare_per_rpk
        self.df.loc[:, "airfare_per_rpk_base"] = airfare_per_rpk_base

        return (marginal_cost_per_rpk, airfare_per_rpk_true, airfare_per_rpk, airfare_per_rpk_base)

`RPKElasticity` (the market-generic cost-feedback layer wired in by
`global.demand.model: cagr_elasticity`) normalises airfare by a fixed
`initial_airfare_per_rpk` scalar. Option (b) needs a *moving* reference price
instead (`airfare_per_rpk_base`, from the fixed supply function above), so it swaps
in this subclass instead.

In [ ]:
class RPKElasticityB(RPKElasticity):
    """Cross-market elasticity layer using a moving base airfare
    (`airfare_per_rpk_base`, from `PassengerAircraftMarginalCostB`) instead of the
    fixed 2019 anchor `RPKElasticity` normally divides by."""

    def __init__(self, name: str, passenger_market_ids: list, *args, **kwargs):
        super().__init__(name=name, passenger_market_ids=passenger_market_ids, *args, **kwargs)
        del self.input_names["initial_airfare_per_rpk"]
        self.input_names["airfare_per_rpk_base"] = pd.Series([0.0])

    def _initialize_df(self):
        super()._initialize_df()
        # airfare_per_rpk_base closes a second cycle through the same
        # marginal-cost discipline as airfare_per_rpk -- both need a seed so
        # GEMSEO's initialization chain can break the strongly-connected group.
        self._coupling_defaults["airfare_per_rpk_base"] = pd.Series(
            self.REFERENCE_AIRFARE_PER_RPK,
            index=range(self.historic_start_year, self.end_year + 1),
        )
        low, high = (
            factor * self.REFERENCE_AIRFARE_PER_RPK for factor in self.AIRFARE_BOUNDS_RELATIVE
        )
        self._coupling_bounds["airfare_per_rpk_base"] = (low, high)

    def compute(self, input_data: dict) -> dict:
        rpk_no_elasticity = input_data["rpk_no_elasticity"]
        airfare_per_rpk = input_data["airfare_per_rpk"]
        price_elasticity = float(input_data["price_elasticity"])
        airfare_base = input_data["airfare_per_rpk_base"]

        elasticity_start = max(
            int(max(int(input_data[f"{mid}_covid_end_year"]) for mid in self.passenger_market_ids))
            + 1,
            self.prospection_start_year,
        )
        proj = slice(elasticity_start, self.end_year)

        multiplier = pd.Series(1.0, index=self.df.index)
        multiplier.loc[proj] = (
            airfare_per_rpk.loc[proj] / airfare_base.loc[proj]
        ) ** price_elasticity

        total_rpk = rpk_no_elasticity * multiplier
        self.df.loc[:, "rpk"] = total_rpk
        self.df.loc[:, "elasticity_factor"] = multiplier

        output_data = {"rpk": total_rpk, "elasticity_factor": multiplier}

        for mid in self.passenger_market_ids:
            rpk_m_base = input_data[f"rpk_{mid}_no_elasticity"]
            rpk_m = rpk_m_base * multiplier
            self.df.loc[:, f"rpk_{mid}"] = rpk_m

            rate_col = f"annual_growth_rate_rpk_{mid}"
            self.df.loc[self.historic_start_year + 1 : self.end_year, rate_col] = (
                rpk_m.pct_change(fill_method=None) * 100
            )

            cagr_m = 100 * (
                (
                    self.df.loc[self.end_year, f"rpk_{mid}"]
                    / self.df.loc[self.prospection_start_year - 1, f"rpk_{mid}"]
                )
                ** (1 / (self.end_year - self.prospection_start_year))
                - 1
            )
            prospective_m = 100 * (
                self.df.loc[self.end_year, f"rpk_{mid}"]
                / self.df.loc[self.prospection_start_year - 1, f"rpk_{mid}"]
                - 1
            )

            output_data[f"rpk_{mid}"] = rpk_m
            output_data[f"annual_growth_rate_rpk_{mid}"] = self.df[rate_col]
            output_data[f"cagr_rpk_{mid}"] = cagr_m
            output_data[f"prospective_evolution_rpk_{mid}"] = prospective_m

        self.df.loc[
            self.historic_start_year + 1 : self.end_year, "annual_growth_rate_passenger"
        ] = total_rpk.pct_change(fill_method=None) * 100
        cagr_rpk = 100 * (
            (
                self.df.loc[self.end_year, "rpk"]
                / self.df.loc[self.prospection_start_year - 1, "rpk"]
            )
            ** (1 / (self.end_year - self.prospection_start_year))
            - 1
        )
        prospective_rpk = 100 * (
            self.df.loc[self.end_year, "rpk"] / self.df.loc[self.prospection_start_year - 1, "rpk"]
            - 1
        )
        output_data["annual_growth_rate_passenger"] = self.df["annual_growth_rate_passenger"]
        output_data["cagr_rpk"] = cagr_rpk
        output_data["prospective_evolution_rpk"] = prospective_rpk

        self._store_outputs(output_data)
        return output_data

`standards` entries (e.g. `models_optim_complex`) load as nested dicts, so a
model swap needs to walk that nesting rather than a single-level
`custom_models=` merge -- see the helper below.

In [ ]:
def replace_nested_model(models_dict, key, new_model):
    """Replace `key` wherever it sits in a (possibly nested) models dict.

    ``custom_models=`` passed to `create_process` only ever adds a *sibling*
    top-level entry: a `standards` group like `models_optim_complex` is itself
    one nested dict, so overriding one of its members needs an in-place walk,
    done here after `create_process` instead.
    """
    if key in models_dict:
        models_dict[key] = new_model
        return True
    for value in models_dict.values():
        if isinstance(value, dict) and replace_nested_model(value, key, new_model):
            return True
    return False

In [ ]:
process_b = create_process(configuration_file="./config_equilibriums.yaml")

process_b.parameters.short_range_energy_per_ask_dropin_fuel_gain_reference_years = []
process_b.parameters.short_range_energy_per_ask_dropin_fuel_gain_reference_years_values = [
    0.0000000001
]
process_b.parameters.medium_range_energy_per_ask_dropin_fuel_gain_reference_years = []
process_b.parameters.medium_range_energy_per_ask_dropin_fuel_gain_reference_years_values = [
    0.0000000001
]
process_b.parameters.long_range_energy_per_ask_dropin_fuel_gain_reference_years = []
process_b.parameters.long_range_energy_per_ask_dropin_fuel_gain_reference_years_values = [
    0.0000000001
]
process_b.parameters.operations_final_gain = 0.0000000001  # [%]
process_b.parameters.operations_start_year = 2025
process_b.parameters.operations_duration = 25.0
process_b.parameters.short_range_load_factor_end_year = 82.39931200000001
process_b.parameters.medium_range_load_factor_end_year = 82.39931200000001
process_b.parameters.long_range_load_factor_end_year = 82.39931200000001
process_b.parameters.carbon_tax_reference_years = [2020, 2034, 2035, 2050]
process_b.parameters.carbon_tax_reference_years_values = [0, 0, 300, 300]

# Swap both halves of the cost-feedback loop for their option-(b) variants, then
# rebuild the discipline list and MDA chain -- `common_setup()` must NOT be
# re-run, or it would re-wire the stock `rpk_elasticity` straight back in.
passenger_ids = [m.id for m in process_b.markets.get(traffic_type="passenger")]
replace_nested_model(
    process_b.models,
    "passenger_aircraft_marginal_cost",
    PassengerAircraftMarginalCostB("passenger_aircraft_marginal_cost"),
)
process_b.models["rpk_elasticity"] = RPKElasticityB(
    name="rpk_elasticity", passenger_market_ids=passenger_ids
)
process_b.disciplines = []
process_b.setup_mda()

In [ ]:
process_b.compute()

In [ ]:
class PassengerAircraftMarginalCostC(AeroMAPSModel):
    """Fixed (average IATA) markup, in EUR/RPK -- option (c)."""

    def __init__(self, name="passenger_aircraft_marginal_cost", fleet_model=None, *args, **kwargs):
        super().__init__(name=name, *args, **kwargs)

    def compute(
        self,
        markup: float,
        total_cost_per_rpk_without_extra_tax: pd.Series,
        total_extra_tax_per_rpk: pd.Series,
        total_subsidy_per_rpk: pd.Series,
    ) -> Tuple[pd.Series, pd.Series]:
        marginal_cost_per_rpk = total_cost_per_rpk_without_extra_tax
        airfare_per_rpk = (
            total_cost_per_rpk_without_extra_tax
            + total_extra_tax_per_rpk
            - total_subsidy_per_rpk
            + markup
        )

        self.df.loc[:, "marginal_cost_per_rpk"] = marginal_cost_per_rpk
        self.df.loc[:, "airfare_per_rpk"] = airfare_per_rpk

        return (marginal_cost_per_rpk, airfare_per_rpk)

In [ ]:
process_c = create_process(configuration_file="./config_equilibriums.yaml")

process_c.parameters.short_range_energy_per_ask_dropin_fuel_gain_reference_years = []
process_c.parameters.short_range_energy_per_ask_dropin_fuel_gain_reference_years_values = [
    0.0000000001
]
process_c.parameters.medium_range_energy_per_ask_dropin_fuel_gain_reference_years = []
process_c.parameters.medium_range_energy_per_ask_dropin_fuel_gain_reference_years_values = [
    0.0000000001
]
process_c.parameters.long_range_energy_per_ask_dropin_fuel_gain_reference_years = []
process_c.parameters.long_range_energy_per_ask_dropin_fuel_gain_reference_years_values = [
    0.0000000001
]
process_c.parameters.operations_final_gain = 0.0000000001  # [%]
process_c.parameters.operations_start_year = 2025
process_c.parameters.operations_duration = 25.0
process_c.parameters.short_range_load_factor_end_year = 82.39931200000001
process_c.parameters.medium_range_load_factor_end_year = 82.39931200000001
process_c.parameters.long_range_load_factor_end_year = 82.39931200000001
process_c.parameters.carbon_tax_reference_years = [2020, 2034, 2035, 2050]
process_c.parameters.carbon_tax_reference_years_values = [0, 0, 300, 300]

replace_nested_model(
    process_c.models,
    "passenger_aircraft_marginal_cost",
    PassengerAircraftMarginalCostC("passenger_aircraft_marginal_cost"),
)
process_c.disciplines = []
process_c.setup_mda()

process_c.parameters.markup = (
    df_t["Avg price (incl. anx), $/RPK"] - df_t["Avg cost, $/RPK"]
).mean()

In [ ]:
process_c.compute()

In [ ]:
class PassengerAircraftMarginalCostD(AeroMAPSModel):
    """Fixed (average IATA) margin rate, applied to total cost -- option (d)."""

    def __init__(self, name="passenger_aircraft_marginal_cost", fleet_model=None, *args, **kwargs):
        super().__init__(name=name, *args, **kwargs)

    def compute(
        self,
        markup: float,
        total_cost_per_rpk_without_extra_tax: pd.Series,
        total_extra_tax_per_rpk: pd.Series,
        total_subsidy_per_rpk: pd.Series,
    ) -> Tuple[pd.Series, pd.Series]:
        marginal_cost_per_rpk = total_cost_per_rpk_without_extra_tax
        airfare_per_rpk = (
            total_cost_per_rpk_without_extra_tax * (1 + markup)
            + total_extra_tax_per_rpk
            - total_subsidy_per_rpk
        )

        self.df.loc[:, "marginal_cost_per_rpk"] = marginal_cost_per_rpk
        self.df.loc[:, "airfare_per_rpk"] = airfare_per_rpk

        return (marginal_cost_per_rpk, airfare_per_rpk)

In [ ]:
process_d = create_process(configuration_file="./config_equilibriums.yaml")

process_d.parameters.short_range_energy_per_ask_dropin_fuel_gain_reference_years = []
process_d.parameters.short_range_energy_per_ask_dropin_fuel_gain_reference_years_values = [
    0.0000000001
]
process_d.parameters.medium_range_energy_per_ask_dropin_fuel_gain_reference_years = []
process_d.parameters.medium_range_energy_per_ask_dropin_fuel_gain_reference_years_values = [
    0.0000000001
]
process_d.parameters.long_range_energy_per_ask_dropin_fuel_gain_reference_years = []
process_d.parameters.long_range_energy_per_ask_dropin_fuel_gain_reference_years_values = [
    0.0000000001
]
process_d.parameters.operations_final_gain = 0.0000000001  # [%]
process_d.parameters.operations_start_year = 2025
process_d.parameters.operations_duration = 25.0
process_d.parameters.short_range_load_factor_end_year = 82.39931200000001
process_d.parameters.medium_range_load_factor_end_year = 82.39931200000001
process_d.parameters.long_range_load_factor_end_year = 82.39931200000001
process_d.parameters.carbon_tax_reference_years = [2020, 2034, 2035, 2050]
process_d.parameters.carbon_tax_reference_years_values = [0, 0, 300, 300]

replace_nested_model(
    process_d.models,
    "passenger_aircraft_marginal_cost",
    PassengerAircraftMarginalCostD("passenger_aircraft_marginal_cost"),
)
process_d.disciplines = []
process_d.setup_mda()

process_d.parameters.markup = df_t["% gross margin"].mean() / 100

In [ ]:
process_d.compute()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update(
    {
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 12,  # Default is usually 10
        "axes.titlesize": 15,  # Subplot titles
        "axes.labelsize": 14,  # Axis labels
        "legend.fontsize": 12,  # Legend text
        "legend.title_fontsize": 13,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
    }
)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)

colors = {"A": "#cb3629", "B": "#092054", "C": "#efbd40", "D": "#7e9b59"}

# =========== RPK ==========
ax[1].plot(process_a.data["vector_outputs"]["rpk"] / 1e9, color=colors["A"])
ax[1].plot(
    process_b.data["vector_outputs"]["rpk"] / 1e9,
    color=colors["B"],
)
ax[1].plot(process_c.data["vector_outputs"]["rpk"] / 1e9, color=colors["C"])
ax[1].plot(process_d.data["vector_outputs"]["rpk"] / 1e9, color=colors["D"])
ax[1].plot(
    process_d.data["vector_outputs"]["rpk_no_elasticity"].loc[2020:2050] / 1e9,
    label="No elasticity",
    linestyle="--",
    color="darkgrey",
)
ax[1].set_title("Revenue Passenger Kilometers (RPK)")
ax[1].set_ylabel("RPK (Bn)")
ax[1].grid(True)
ax[1].set_xlim(2024, 2050)
ax[1].set_title("Revenue Passenger Kilometers", y=-0.14)

# =========== AIRFARE ==========
# Dotted lines: the DOC-based cost baseline (total_cost_per_rpk_without_extra_tax +
# total_extra_tax_per_rpk - total_subsidy_per_rpk), the same quantity all four
# marginal-cost models start from before adding their own supply/markup logic.
ax[0].plot(
    process_a.data["vector_outputs"]["airfare_per_rpk"],
    label="Adjusted supply",
    color=colors["A"],
)
ax[0].plot(
    process_b.data["vector_outputs"]["airfare_per_rpk"],
    label="Adjusted demand",
    color=colors["B"],
)
ax[0].plot(process_c.data["vector_outputs"]["airfare_per_rpk"], label="Markup", color=colors["C"])
ax[0].plot(
    process_d.data["vector_outputs"]["airfare_per_rpk"], label="Markup rate", color=colors["D"]
)

for label, process in (("A", process_a), ("B", process_b), ("C", process_c), ("D", process_d)):
    cost_baseline = (
        process.data["vector_outputs"]["total_cost_per_rpk_without_extra_tax"].loc[2020:2050]
        + process.data["vector_outputs"]["total_extra_tax_per_rpk"].loc[2020:2050]
        - process.data["vector_outputs"]["total_subsidy_per_rpk"].loc[2020:2050]
    )
    ax[0].plot(cost_baseline, linestyle=":", color=colors[label])

ax[0].set_title("Airfare per RPK")
ax[0].set_ylabel("Airfare (EUR/RPK)")
ax[0].grid(True)
ax[0].set_xlim(2024, 2050)
ax[0].set_title("Airfare per RPK", y=-0.14)  # Subtitle below the subplot

handles1, labels1 = ax[0].get_legend_handles_labels()
model_handles = handles1[:4]  # First 4 are the airfare lines
model_labels = labels1[:4]
legend1 = ax[0].legend(
    handles=model_handles,
    labels=model_labels,
    title="Supply models",
    framealpha=1,
    loc="upper left",
)
legend1.get_title().set_color("black")

# Second legend: Costs
cost_line = ax[0].plot(np.nan, np.nan, linestyle=":", color="black", label="Costs")[0]
af_line = ax[0].plot(np.nan, np.nan, linestyle="-", color="black", label="Airfare")[0]
legend2 = ax[0].legend(
    handles=[cost_line, af_line],
    loc="lower right",
    framealpha=1,
)
legend2.get_title().set_color("black")

ax[0].add_artist(legend1)

ax[1].legend()

plt.savefig("supply.pdf")